# Imports

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
import pandas as pd
import numpy as np
from plotnine import *
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
# import tensorflow_probability as tfp
import keras_opt
from keras_opt import scipy_optimizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder

# Data

In [3]:
data2 = pd.read_csv('../datasets/features_with_next_prices_clean.csv')
data2.dropna(inplace = True)
data2.reset_index(inplace = True)
data2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76469 entries, 0 to 76468
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   index                        76469 non-null  int64  
 1   date                         76469 non-null  object 
 2   return                       76469 non-null  float64
 3   high_open_ratio              76469 non-null  float64
 4   open_low_ratio               76469 non-null  float64
 5   high_close_ratio             76469 non-null  float64
 6   volatility_by_candle_number  76469 non-null  float64
 7   weekday                      76469 non-null  int64  
 8   volume                       76469 non-null  float64
 9   candle_direction             76469 non-null  float64
 10  earning_moment               76469 non-null  int64  
 11  rolling_std_return           76469 non-null  float64
 12  true_range                   76469 non-null  float64
 13  ATR             

In [5]:
X = data2.drop(columns=['index', 'date', 'candle_direction', 'tokenId', 'open_next', 'close_next'])
y = data2['open_next'].values.reshape(-1, 1)
contin = ['volatility_by_candle_number', 'volume',
          'rolling_std_return', 'true_range', 'ATR', 'volume_delta', 'volume_per_range', 'vix', 
          'amzn_vix_proxy', 'iv_delta', 'put_spread', 'call_spread', 'rsi', 'macd', 'return']
categorical = ['weekday', 'earning_moment', 'volume_spike']


preprocessor = make_column_transformer(
    (OneHotEncoder(sparse_output=False, drop="first"), categorical),
    remainder="passthrough"
)

tf.keras.backend.set_floatx('float64')
# X_train, X_test, y_train_raw, y_test_raw = train_test_split(X, y, test_size=0.2, random_state=3619) 
n = len(X)
split_idx = int(n * 0.8)   # 80% train / 20% test for chronological split



X_train_raw = X[:split_idx]
X_test_raw  = X[split_idx:]
y_train_raw = y[:split_idx]
y_test_raw  = y[split_idx:]



scaler = StandardScaler()
y_train = scaler.fit_transform(y_train_raw)
y_test = scaler.transform(y_test_raw)
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)


X_train64 = X_train.astype('float64')
X_test64 = X_test.astype('float64')
y_train64 = y_train.astype('float64')
y_test64 = y_test.astype('float64')
num_features = X_train.shape[1]

# Feedforward

## Adam

In [5]:
tf.keras.backend.clear_session() # if necessary to remove float64
tf.keras.backend.set_floatx('float32')

In [6]:
model3 = keras.Sequential([ 
    keras.layers.Dense(512, activation='relu', input_shape=(num_features,)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(1, activation='linear')
])

model3.compile(loss='mean_squared_error',
               optimizer='adam',
               metrics=['mae'])

history_adam_stocks = model3.fit(X_train, y_train,
                           epochs=20, verbose=1, batch_size=256, # larger batch size for larger datasets... right?
                           validation_data=(X_test, y_test))

history_adam = history_adam_stocks.history

Epoch 1/20
239/239 [==============================] - 2s 5ms/step - loss: 0.0239 - mae: 0.0801 - val_loss: 0.0027 - val_mae: 0.0308
Epoch 2/20
239/239 [==============================] - 1s 4ms/step - loss: 0.0031 - mae: 0.0366 - val_loss: 0.0035 - val_mae: 0.0459
Epoch 3/20
239/239 [==============================] - 1s 4ms/step - loss: 0.0023 - mae: 0.0308 - val_loss: 0.0012 - val_mae: 0.0186
Epoch 4/20
239/239 [==============================] - 1s 4ms/step - loss: 0.0020 - mae: 0.0293 - val_loss: 0.0011 - val_mae: 0.0176
Epoch 5/20
239/239 [==============================] - 1s 5ms/step - loss: 0.0019 - mae: 0.0274 - val_loss: 8.6395e-04 - val_mae: 0.0138
Epoch 6/20
239/239 [==============================] - 1s 5ms/step - loss: 0.0016 - mae: 0.0243 - val_loss: 8.3643e-04 - val_mae: 0.0149
Epoch 7/20
239/239 [==============================] - 1s 5ms/step - loss: 0.0014 - mae: 0.0230 - val_loss: 0.0010 - val_mae: 0.0154
Epoch 8/20
239/239 [==============================] - 1s 6ms/step - 

## L-BFGS (via keras-opt-mod)

In [7]:
tf.keras.backend.clear_session()
tf.keras.backend.set_floatx('float64')

In [8]:
with tf.device('/CPU:0'): # needed bcs I cannot change GPU settings during runtime
    model4 = keras.Sequential([ 
        keras.layers.Dense(256, activation='relu', input_shape=(num_features,)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1, activation='linear')
    ])

    model4.compile(loss='mean_squared_error', 
                metrics=['mae'])

    model4.train_function = scipy_optimizer.make_train_function(
        model4,
        method="L-BFGS-B",            # top-level OK
        maxiter=300,                  # top-level OK
    )

    history_lfbgs_stocks = model4.fit(X_train64, y_train64,
                            epochs=1, verbose=1, batch_size=2048,
                            validation_data=(X_test64, y_test64))
    second_order_stocks = history_lfbgs_stocks.history

     24/Unknown - 0s 10ms/step - loss: 1.0123

2025-11-17 14:30:41.118460: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


     26/Unknown - 0s 13ms/step - loss: 1.1244

2025-11-17 14:30:41.770888: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


     27/Unknown - 0s 8ms/step - loss: 0.4493

...

     26/Unknown - 0s 9ms/step - loss: 0.0034

2025-11-17 14:31:10.981623: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


     27/Unknown - 1s 20ms/step - loss: 0.0023

2025-11-17 14:31:42.303967: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


30/30 [==============================] - 116s 4s/step - loss: 0.0017 - mae: 0.0229 - val_loss: 5.0108e-04 - val_mae: 0.0121


## Trust-NCG (via keras-opt-mod)

In [12]:
with tf.device('/CPU:0'): # needed bcs I cannot change GPU settings during runtime
    model5 = keras.Sequential([ 
        keras.layers.Dense(256, activation='relu', input_shape=(num_features,)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1, activation='linear')
    ])

    model5.compile(loss='mean_squared_error', 
                metrics=['mae'])

    model5.train_function = scipy_optimizer.make_train_function(
        model5,
        method="trust-ncg",            # top-level OK
        maxiter=300,
        gtol=1e-05,
        xtol=1e-08
    )

    history_ncg_stocks = model5.fit(X_train64, y_train64,
                            epochs=1, verbose=1, 
                            batch_size=2048,
                            validation_data=(X_test64, y_test64))
    trust_ncg_stocks = history_ncg_stocks.history

     24/Unknown - 0s 10ms/step - loss: 52.6669

2025-11-17 14:35:24.844224: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


     25/Unknown - 0s 9ms/step - loss: 0.0025 

2025-11-17 14:39:36.932592: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


     24/Unknown - 0s 10ms/step - loss: 0.0014

2025-11-17 14:48:25.253423: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


     26/Unknown - 0s 9ms/step - loss: 0.0012 

/Users/kavinravi/Documents/CPSC_393/.venv311/lib/python3.11/site-packages/scipy/optimize/_minimize.py:806: RuntimeWarning: A bad approximation caused failure to predict improvement.


         Current function value: 0.001243
         Iterations: 211
         Function evaluations: 213
         Gradient evaluations: 174
         Hessian evaluations: 1507
30/30 [==============================] - 1843s 61s/step - loss: 0.0012 - mae: 0.0202 - val_loss: 9.0097e-04 - val_mae: 0.0150


## Performance Metrics

In [15]:
print("Adam Metrics:")
print(f'Training Loss: {history_adam["loss"][-1]}')
print(f'Training MAE: {history_adam["mae"][-1]}')
print(f'Validation Loss: {history_adam["val_loss"][-1]}')
print(f'Validation MAE: {history_adam["val_mae"][-1]}')
print("\n")
print("L-BFGS Metrics:")
print(f'Training Loss: {second_order_stocks["loss"]}')
print(f'Training MAE: {second_order_stocks["mae"]}')
print(f'Validation Loss: {second_order_stocks["val_loss"]}')
print(f'Validation MAE: {second_order_stocks["val_mae"]}')
print("\n")
print("Trust-NCG Metrics:")
print(f'Training Loss: {trust_ncg_stocks["loss"]}')
print(f'Training MAE: {trust_ncg_stocks["mae"]}')
print(f'Validation Loss: {trust_ncg_stocks["val_loss"]}')
print(f'Validation MAE: {trust_ncg_stocks["val_mae"]}')

Adam Metrics:
Training Loss: 0.0005298185481415426
Training MAE: 0.014447620035095999
Validation Loss: 0.001457599530479681
Validation MAE: 0.018008241495617756


L-BFGS Metrics:
Training Loss: [0.0013619952219976683]
Training MAE: [0.020504205228201223]
Validation Loss: [0.0007475113156704467]
Validation MAE: [0.01326425398072456]


Trust-NCG Metrics:
Training Loss: [0.00112405778709223]
Training MAE: [0.019236584499169308]
Validation Loss: [0.0008706102926179692]
Validation MAE: [0.014398636751897015]


## Predicted vs Actual Values

In [ ]:
y_pred_adam   = model3.predict(X_test.astype(np.float32))        # likely float32
y_pred_lbfgs  = model4.predict(X_test64)                          # float64
y_pred_ncg    = model5.predict(X_test64)                          # float64

# inverse transform so values are in original scale
y_test_orig = scaler.inverse_transform(y_test64.reshape(-1,1)).ravel()
y_pred_adam_orig = scaler.inverse_transform(y_pred_adam.astype(np.float64)).ravel()
y_pred_lbfgs_orig = scaler.inverse_transform(y_pred_lbfgs).ravel()
y_pred_ncg_orig = scaler.inverse_transform(y_pred_ncg).ravel()

N = len(y_test_orig)
idx = np.arange(N)
# pick first n samples to plot
n_plot = 500
idx_sub = idx[:n_plot]

plt.figure(figsize=(10,5))
plt.plot(idx_sub, y_test_orig[idx_sub], label='Actual', linewidth=1)
plt.plot(idx_sub, y_pred_adam_orig[idx_sub], label='Predicted – Adam', linewidth=1, linestyle='--')
plt.plot(idx_sub, y_pred_lbfgs_orig[idx_sub], label='Predicted – L-BFGS', linewidth=1, linestyle=':')
plt.plot(idx_sub, y_pred_ncg_orig[idx_sub], label='Predicted – Trust-NCG', linewidth=1, linestyle='--')
plt.legend()
plt.title("Actual vs Predicted (first {} samples)".format(n_plot))
plt.xlabel("Test sample index")
plt.ylabel("Target value")
plt.show()

plt.figure(figsize=(10,5))
plt.plot(idx_sub, y_test[idx_sub], label='Actual', linewidth=1)
plt.plot(idx_sub, y_pred_adam[idx_sub], label='Predicted – Adam', linewidth=1, linestyle='--')
plt.plot(idx_sub, y_pred_lbfgs[idx_sub], label='Predicted – L-BFGS', linewidth=1, linestyle=':')
plt.plot(idx_sub, y_pred_ncg[idx_sub], label='Predicted – Trust-NCG', linewidth=1, linestyle='--')
plt.legend()
plt.title("Actual vs Predicted (first {} samples) (SCALED)".format(n_plot))
plt.xlabel("Test sample index")
plt.ylabel("Scaled Target value")
plt.show()

# LSTM

In [6]:
def create_sequences(X, y, seq_length):
    num_sequences = len(X) - seq_length
    num_features = X.shape[1] if len(X.shape) > 1 else 1
    
    # Pre-allocate arrays for memory efficiency
    X_seq = np.zeros((num_sequences, seq_length, num_features))
    y_seq = np.zeros(num_sequences)
    
    for i in range(num_sequences):
        X_seq[i] = X[i:i+seq_length]
        y_seq[i] = y[i+seq_length]
    
    return X_seq, y_seq

# Define sequence length
SEQ_LEN = 78

# Create sequences from your already-scaled feedforward data
X_train_lstm, y_train_lstm = create_sequences(X_train64, y_train64, SEQ_LEN)
X_test_lstm, y_test_lstm = create_sequences(X_test64, y_test64, SEQ_LEN)

print(f"Original shapes: X_train64={X_train64.shape}, y_train64={y_train64.shape}")
print(f"Sequence shapes: X_train_lstm={X_train_lstm.shape}, y_train_lstm={y_train_lstm.shape}")
print(f"num_features={num_features}")

/var/folders/44/_9r3lfrd4h1bt3rsqqzzs7sm0000gn/T/ipykernel_62363/1120672623.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)


Original shapes: X_train64=(61175, 25), y_train64=(61175, 1)
Sequence shapes: X_train_lstm=(61097, 78, 25), y_train_lstm=(61097,)
num_features=25


## Adam

In [14]:
model_lstm = keras.Sequential([
    keras.layers.LSTM(64, return_sequences=False, input_shape=(SEQ_LEN, num_features)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1)
])

model_lstm.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# train
history_lstm = model_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_lstm, y_test_lstm),
    verbose=1
)

# # predict
# y_pred_lstm = model_lstm.predict(X_test)
# print("Predictions generated.")

Epoch 1/10
955/955 [==============================] - 50s 51ms/step - loss: 0.0099 - mae: 0.0464 - val_loss: 2.2205e-04 - val_mae: 0.0103
Epoch 2/10
955/955 [==============================] - 46s 49ms/step - loss: 0.0022 - mae: 0.0267 - val_loss: 3.1000e-04 - val_mae: 0.0140
Epoch 3/10
955/955 [==============================] - 47s 49ms/step - loss: 0.0016 - mae: 0.0230 - val_loss: 1.8785e-04 - val_mae: 0.0098
Epoch 4/10
955/955 [==============================] - 47s 49ms/step - loss: 0.0012 - mae: 0.0203 - val_loss: 1.7791e-04 - val_mae: 0.0095
Epoch 5/10
955/955 [==============================] - 48s 51ms/step - loss: 0.0010 - mae: 0.0184 - val_loss: 1.6417e-04 - val_mae: 0.0096
Epoch 6/10
955/955 [==============================] - 47s 49ms/step - loss: 8.3685e-04 - mae: 0.0168 - val_loss: 1.7103e-04 - val_mae: 0.0090
Epoch 7/10
955/955 [==============================] - 46s 48ms/step - loss: 7.2394e-04 - mae: 0.0158 - val_loss: 1.7709e-04 - val_mae: 0.0094
Epoch 8/10
955/955 [======

## L-BFGS

In [ ]:
with tf.device('/CPU:0'):
    model_lstm_lbfgs = keras.Sequential([
        keras.layers.LSTM(64, return_sequences=False, input_shape=(SEQ_LEN, num_features)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1)
    ])

    model_lstm_lbfgs.compile(
        loss='mse',
        metrics=['mae']
    )

    model_lstm_lbfgs.train_function = scipy_optimizer.make_train_function(
        model_lstm_lbfgs,
        method="L-BFGS-B",
        maxiter=300
    )
    # train
    history_lstm_lbfgs = model_lstm_lbfgs.fit(
        X_train_lstm, y_train_lstm,
        epochs=1,
        batch_size=64, # must be much smaller than ffn due to memory constraints, increase as available
        validation_data=(X_test_lstm, y_test_lstm),
        verbose=1
    )
    lbfgs_lstm = history_lstm_lbfgs.history

    954/Unknown - 193s 203ms/step - loss: 1.0647

2025-11-18 17:02:12.650959: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


    954/Unknown - 197s 206ms/step - loss: 0.7186

2025-11-18 17:05:29.322495: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


    954/Unknown - 196s 205ms/step - loss: 0.1399

...

      0/Unknown - 0s 0s/step - loss: 0.0052.0067

2025-11-18 18:27:52.236551: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


    954/Unknown - 199s 208ms/step - loss: 0.0040

2025-11-18 19:34:48.139862: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


    184/Unknown - 2398s 13s/step - loss: 0.0033

## Trust-NCG

In [ ]:
with tf.device('/CPU:0'):
    model_lstm = keras.Sequential([
        keras.layers.LSTM(64, return_sequences=False, input_shape=(SEQ_LEN, num_features)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1)
    ])

    model_lstm.compile(
        loss='mse',
        metrics=['mae']
    )

    model_lstm.train_function = scipy_optimizer.make_train_function(
        model_lstm,
        method="L-BFGS-B",
        maxiter=300,
        gtol=1e-05,
        xtol=1e-08
    )
    # train
    history_lstm = model_lstm.fit(
        X_train_lstm, y_train_lstm,
        epochs=1,
        batch_size=2048,
        validation_data=(X_test_lstm, y_test_lstm),
        verbose=1
    )
    lbfgs_lstm = history_lstm.history